[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Migrations


## What you will be able to do

Change documents you have already written. Generate a migration with `beanie new-migration`, fill
in the two empty classes it gives you, and run it with `beanie migrate`. Say why the command is a
subprocess and what that means for where your models have to live. Choose between an iterative
migration and a free fall one. And recognize the four ways the command fails, the first of which is
not about the database at all.


## The idea

### The problem

A collection has no schema, so adding a field to a model changes nothing that is already stored.
Every document written before the change still has the old shape, and **Beanie Documents** showed
what that does on the next read: a `ValidationError` for a field that is simply not there.

A default hides it. A migration fixes it, by writing the field onto the documents that lack it.

### What a Beanie migration is

A file with two classes in it, `Forward` and `Backward`, each holding methods decorated with
`@iterative_migration()` or `@free_fall_migration()`. `beanie migrate` imports the files in order,
runs the ones not yet recorded, and writes each name into a `migrations_log` collection so it is not
run twice.

### Why it runs as a subprocess

Because it is a command line tool: it starts a fresh Python, imports your migration files, and
imports whatever they import. A model defined in a notebook cell does not exist in that
interpreter, which is why the migration file below either defines its models or imports them from a
file on disk.

### Where this shows up

The second release. The first one defines the models and the second one changes them, and by then
there is data.

### What this notebook covers

`new-migration` and what it generates. Writing a `Forward`. `migrate`, and the log it keeps.
Iterative against free fall. Then the failures: the models it cannot import, the decorator with no
`input_document`, the missing `-db`, and the transaction it wants a replica set for.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import pathlib
import subprocess
import sys

import pymongo

WORK = pathlib.Path("/tmp/guide_migrations")
(WORK / "migrations").mkdir(parents=True, exist_ok=True)

(WORK / "migrations" / "20260101000000_add_greeting.py").write_text('''
from beanie import Document, iterative_migration


class Person(Document):
    name: str

    class Settings:
        name = "mig_people"


class Greeted(Document):
    name: str
    greeting: str = ""

    class Settings:
        name = "mig_people"


class Forward:
    @iterative_migration()
    async def add_greeting(self, input_document: Person, output_document: Greeted):
        output_document.greeting = f"hello {input_document.name}"


class Backward: ...
''')

shop = pymongo.MongoClient("mongodb://127.0.0.1:27017/shop").get_default_database()
shop.mig_people.drop()
shop.migrations_log.drop()
shop.mig_people.insert_many([{"name": "ana"}, {"name": "bo"}])
print("before:", list(shop.mig_people.find({}, {"_id": 0})))

subprocess.run([sys.executable, "-m", "beanie.executors.migrate", "migrate",
                "-uri", "mongodb://127.0.0.1:27017", "-db", "shop", "-p", "migrations"],
               cwd=WORK, capture_output=True, text=True)

print("after: ", list(shop.mig_people.find({}, {"_id": 0})))
print("a field that no document had, added to every one of them")
```

```
before: [{'name': 'ana'}, {'name': 'bo'}]
after:  [{'name': 'ana', 'greeting': 'hello ana'}, {'name': 'bo', 'greeting': 'hello bo'}]
a field that no document had, added to every one of them
```

Two models over one collection, `Person` as it was and `Greeted` as it should be, and a function
that turns one into the other. Beanie reads every document as the first, hands it to you as the
second, and writes back what you changed.


## Setup

Ten imports, MongoDB, the boot cell, and a folder on disk.

- `pymongo` reads the collection so the notebook can show what a migration did
- `subprocess` and `sys` run the `beanie` command, `pathlib` and `shutil` manage its folder
- `subprocess`, `os`, `time`, `random`, `version` and `PackageNotFoundError` run the boot cell

`fresh_migrations` empties `/tmp/guide_migrations`, `write_migration` puts one file in it with a
name of our choosing, and `run_beanie` runs the command and returns its exit code and output.
`reset_people` puts two documents and an empty migration log back, so every section starts from the
same place and this notebook can be run from the top as often as you like.

Note `migrations_log`: that is the collection Beanie records applied migrations in, and clearing it
is what makes a migration runnable a second time.


In [1]:
import os
import pathlib
import random
import shutil
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

WORK = pathlib.Path("/tmp/guide_migrations")                        # a folder on disk, not in here
STANDALONE = "mongodb://127.0.0.1:27018"                            # the plain mongod, for one error


def fresh_migrations():
    """An empty migrations folder. The names are ours, so the outputs below are reproducible."""
    shutil.rmtree(WORK, ignore_errors=True)
    (WORK / "migrations").mkdir(parents=True)
    return WORK


def write_migration(filename, body):
    """Write one migration file. beanie new-migration would name it after the clock."""
    path = WORK / "migrations" / filename
    path.write_text(body.strip() + "\n")
    return path.name


def run_beanie(*arguments, cwd=None):
    """Run the beanie command as a subprocess and hand back its exit code and last lines.

    It is a subprocess because that is what the command is: it imports your migration files in a
    fresh interpreter, which is why models defined in a notebook cell are invisible to it."""
    done = subprocess.run([sys.executable, "-m", "beanie.executors.migrate", *arguments],
                          cwd=cwd or WORK, capture_output=True, text=True)
    output = (done.stdout + done.stderr).strip().splitlines()
    return done.returncode, output


def people(port=27017):
    with pymongo.MongoClient(f"mongodb://127.0.0.1:{port}/shop") as client:
        shop = client.get_default_database()
        return [dict(sorted(row.items())) for row in
                shop.mig_people.find({}, {"_id": 0}).sort("name")]


def reset_people(port=27017):
    """Two documents and no migration history, so every section starts from the same place."""
    with pymongo.MongoClient(f"mongodb://127.0.0.1:{port}/shop") as client:
        shop = client.get_default_database()
        shop.mig_people.drop()
        shop.migrations_log.drop()                                  # not "migrations": the log
        shop.mig_people.insert_many([{"name": "ana"}, {"name": "bo"}])
        return shop.mig_people.count_documents({})


ADD_GREETING = """
from beanie import Document, iterative_migration


class Person(Document):
    name: str

    class Settings:
        name = "mig_people"


class Greeted(Document):
    name: str
    greeting: str = ""

    class Settings:
        name = "mig_people"


class Forward:
    @iterative_migration()
    async def add_greeting(self, input_document: Person, output_document: Greeted):
        output_document.greeting = f"hello {input_document.name}"


class Backward:
    @iterative_migration()
    async def drop_greeting(self, input_document: Greeted, output_document: Person):
        pass
"""


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print("work:   ", fresh_migrations())
print("people: ", reset_people(), "documents")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
work:    /tmp/guide_migrations
people:  2 documents
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


## Worked examples

### What new-migration gives you

The real command, with its real output:


In [2]:
fresh_migrations()
code_out, output = run_beanie("new-migration", "-n", "add_greeting", "-p", "migrations")

generated = sorted(path.name for path in (WORK / "migrations").glob("*.py"))
print("exit code:", code_out)
print("the name ends with:", generated[0].split("_", 1)[1])
print("and begins with a timestamp, which is why this notebook writes its own names")
print()
print("what is in it:")
print((WORK / "migrations" / generated[0]).read_text().strip())


exit code: 0
the name ends with: add_greeting.py
and begins with a timestamp, which is why this notebook writes its own names

what is in it:
class Forward: ...


class Backward: ...


Two empty class bodies. Everything else is yours: the imports, the models, the decorator and the
function. The command's only job is the filename, whose timestamp prefix is what orders the
migrations.

### Writing one

The file needs the models it works on, which is why it imports `Document` and declares them:


In [3]:
fresh_migrations()
name = write_migration("20260101000000_add_greeting.py", ADD_GREETING)

print("wrote:", name)
print()
for line in ADD_GREETING.strip().splitlines()[:16]:
    print("  " + line)
print("  ...")


wrote: 20260101000000_add_greeting.py

  from beanie import Document, iterative_migration
  
  
  class Person(Document):
      name: str
  
      class Settings:
          name = "mig_people"
  
  
  class Greeted(Document):
      name: str
      greeting: str = ""
  
      class Settings:
          name = "mig_people"
  ...


`Person` is the shape the documents have now and `Greeted` is the shape they should have. Both name
the same collection in their `Settings`, which is the part that makes this a migration rather than
a copy between two collections.

### Running it


In [4]:
print("before:", people())
reset_people()

exit_code, output = run_beanie("migrate", "-uri", "mongodb://127.0.0.1:27017",
                               "-db", "shop", "-p", "migrations")
print("exit code:", exit_code)
for line in output:
    print("  " + line)

print("after: ", people())


before: [{'name': 'ana'}, {'name': 'bo'}]
exit code: 0
  Building migration list
  Running migrations forward without limit
  Running migration add_greeting from module 20260101000000_add_greeting.py
after:  [{'greeting': 'hello ana', 'name': 'ana'}, {'greeting': 'hello bo', 'name': 'bo'}]


### The log, and why it does not run twice


In [5]:
with pymongo.MongoClient(URI) as client:
    log = [row["name"] for row in client.get_default_database().migrations_log.find()]
print("recorded:", log)

exit_code, output = run_beanie("migrate", "-uri", "mongodb://127.0.0.1:27017",
                               "-db", "shop", "-p", "migrations")
print("running it again:", output[-1] if output else "(nothing to do)")
print("documents unchanged:", people())


recorded: ['20260101000000_add_greeting.py']
running it again: Building migration list
documents unchanged: [{'greeting': 'hello ana', 'name': 'ana'}, {'greeting': 'hello bo', 'name': 'bo'}]


The log is a collection like any other, in the same database, and that is where "which migrations
have run" lives. Dropping it makes every migration runnable again, which is what `reset_people`
does and is not something to do to a real database.

### Iterative against free fall

`@iterative_migration()` gives you one document at a time, as two models, and writes back what
changed. `@free_fall_migration()` gives you the raw collection and gets out of the way:


In [6]:
FREE_FALL = """
from beanie import Document, free_fall_migration


class Person(Document):
    name: str
    greeting: str = ""

    class Settings:
        name = "mig_people"


class Forward:
    @free_fall_migration(document_models=[Person])
    async def shout(self, session):
        await Person.get_pymongo_collection().update_many(
            {}, [{"$set": {"greeting": {"$toUpper": "$greeting"}}}], session=session)


class Backward: ...
"""

reset_people()
run_beanie("migrate", "-uri", "mongodb://127.0.0.1:27017", "-db", "shop", "-p", "migrations")
print("after the iterative one:", people())

write_migration("20260102000000_shout.py", FREE_FALL)
run_beanie("migrate", "-uri", "mongodb://127.0.0.1:27017", "-db", "shop", "-p", "migrations")
print("after the free fall one:", people())


after the iterative one: [{'greeting': 'hello ana', 'name': 'ana'}, {'greeting': 'hello bo', 'name': 'bo'}]
after the free fall one: [{'greeting': 'HELLO ANA', 'name': 'ana'}, {'greeting': 'HELLO BO', 'name': 'bo'}]


The free fall version did the whole collection in one `update_many` on the server, which is
enormously faster over a large collection and gives you no validation at all. The iterative one read
and wrote every document through a model, which is slower and catches the document that does not fit.

Use iterative when the change is per document and the models express it. Use free fall when the
change is one server-side operation, and accept that you are back to writing MongoDB by hand.

Both are handed a `session`, which is what makes the whole migration one transaction.

### When to reach for which

| What you want | How to write it |
|---|---|
| a new, empty migration | `beanie new-migration -n name -p migrations` |
| a change per document, validated | `@iterative_migration()`, two models |
| one server-side operation | `@free_fall_migration(document_models=[...])` |
| to run them | `beanie migrate -uri ... -db ... -p migrations` |
| to undo one | `beanie migrate --backward -d 1 ...` |
| against a plain mongod | add `--no-use-transaction`, and lose the rollback |
| to know what has run | the `migrations_log` collection |

The default is iterative, because it validates and because most changes really are per document.
Reach for free fall when iterating would mean reading a million documents to set one field.

### A migration that can be undone, finished


In [7]:
REVERSIBLE = """
from beanie import Document, iterative_migration


class Before(Document):
    name: str
    greeting: str = ""

    class Settings:
        name = "mig_people"


class After(Document):
    name: str
    greeting: str = ""
    initial: str = ""

    class Settings:
        name = "mig_people"


class Forward:
    @iterative_migration()
    async def add_initial(self, input_document: Before, output_document: After):
        output_document.initial = input_document.name[:1].upper()


class Backward:
    @iterative_migration()
    async def drop_initial(self, input_document: After, output_document: Before):
        output_document.greeting = input_document.greeting
"""

fresh_migrations()
reset_people()
write_migration("20260103000000_add_initial.py", REVERSIBLE)

run_beanie("migrate", "-uri", "mongodb://127.0.0.1:27017", "-db", "shop", "-p", "migrations")
print("forward: ", people())

run_beanie("migrate", "--backward", "-d", "1", "-uri", "mongodb://127.0.0.1:27017",
           "-db", "shop", "-p", "migrations")
print("backward:", people())


forward:  [{'greeting': '', 'initial': 'A', 'name': 'ana'}, {'greeting': '', 'initial': 'B', 'name': 'bo'}]
backward: [{'greeting': '', 'name': 'ana'}, {'greeting': '', 'name': 'bo'}]


The `Backward` class is what makes this reversible, and it is the part everybody leaves as `...`.
Writing it costs a few minutes and is the difference between a bad migration being an inconvenience
and being an incident.

Note that going backward is not automatic: Beanie cannot infer how to undo a change, so `Backward`
is a migration in its own right, written by you, in the opposite direction.

### Where each part came from

| In the migration | What it relies on | The section that showed it |
|---|---|---|
| two models, one collection | `Settings.name` naming the same collection | Writing one |
| `@iterative_migration()` | one document at a time, through models | Iterative against free fall |
| `input_document` and `output_document` | the decorator reading the signature | Writing one |
| the `Backward` class | nothing being inferred | A migration that can be undone |
| `migrations_log` | each migration running once | The log |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/15-migrations-solutions.ipynb).

**1.** Generate a migration and print what is inside it.


In [8]:
# your code here


**2.** Write an iterative migration that adds a field, and run it.


In [9]:
# your code here


**3.** Show that running it a second time does nothing.


In [10]:
# your code here


**4.** Write a free fall migration that changes every document in one operation.


In [11]:
# your code here


**5.** Run a migration backward.


In [12]:
# your code here


**6.** Run a migration against the standalone on port 27018.


In [13]:
# your code here


## Common errors

### ModuleNotFoundError: the models the subprocess cannot see


In [14]:
NEEDS_AN_IMPORT = """
from beanie import iterative_migration
from models import Person, Greeted                                  # a module that is not there


class Forward:
    @iterative_migration()
    async def add_greeting(self, input_document: Person, output_document: Greeted):
        output_document.greeting = "hello"


class Backward: ...
"""

fresh_migrations()
reset_people()
write_migration("20260101000000_needs_import.py", NEEDS_AN_IMPORT)

exit_code, output = run_beanie("migrate", "-uri", "mongodb://127.0.0.1:27017",
                               "-db", "shop", "-p", "migrations")
print("exit code:", exit_code)
print("  ", [line for line in output if "Error" in line][-1])


exit code: 1
   ModuleNotFoundError: No module named 'models'


The first thing that breaks is not the database. `beanie migrate` starts a new interpreter and
imports your migration file, and anything that file imports has to be importable from the folder it
runs in.

Models defined in a notebook cell do not exist there at all, which is why a notebook that uses
migrations has to write its models to a file:


In [15]:
(WORK / "models.py").write_text("""
from beanie import Document


class Person(Document):
    name: str

    class Settings:
        name = "mig_people"


class Greeted(Document):
    name: str
    greeting: str = ""

    class Settings:
        name = "mig_people"
""")

exit_code, output = run_beanie("migrate", "-uri", "mongodb://127.0.0.1:27017",
                               "-db", "shop", "-p", "migrations")
print("exit code:", exit_code, "| documents:", people())


exit code: 0 | documents: [{'greeting': 'hello', 'name': 'ana'}, {'greeting': 'hello', 'name': 'bo'}]


The file sits next to the migrations folder and the subprocess runs with that folder as its working
directory, so `from models import Person` finds it. In a real project this is your application's
models module, imported the same way your application imports it.

### RuntimeError: input_signature must not be None


In [16]:
NO_INPUT = """
from beanie import Document, iterative_migration


class Person(Document):
    name: str

    class Settings:
        name = "mig_people"


class Forward:
    @iterative_migration()
    async def add_greeting(self):                                   # no input_document
        pass


class Backward: ...
"""

fresh_migrations()
reset_people()
write_migration("20260101000000_no_input.py", NO_INPUT)

exit_code, output = run_beanie("migrate", "-uri", "mongodb://127.0.0.1:27017",
                               "-db", "shop", "-p", "migrations")
print("exit code:", exit_code)
print("  ", [line for line in output if "RuntimeError" in line][-1])


exit code: 1
   RuntimeError: input_signature must not be None


`@iterative_migration()` reads the function's annotations to find out which model to read documents
as and which to write them back as. With no `input_document` parameter there is nothing to read, and
it says so when the file is imported rather than when the migration runs.

The parameter names matter as well as the annotations: they must be `input_document` and
`output_document`.

### OperationFailure: Transaction numbers are only allowed on a replica set member


In [17]:
fresh_migrations()
reset_people(port=27018)                                            # the plain mongod
write_migration("20260101000000_add_greeting.py", ADD_GREETING)

exit_code, output = run_beanie("migrate", "-uri", STANDALONE, "-db", "shop", "-p", "migrations")
print("exit code:", exit_code)
print("  ", [line for line in output if "OperationFailure" in line][-1].split(", full error")[0])


exit code: 1
   pymongo.errors.OperationFailure: Transaction numbers are only allowed on a replica set member or mongos


`beanie migrate` wraps the whole migration in a transaction so that a failure halfway leaves
nothing applied, and a transaction needs a replica set. This is the same refusal
**Bulk Writes and Transactions** met, arriving from a command line tool.

There is an escape, and it costs exactly what you would expect:


In [18]:
exit_code, output = run_beanie("migrate", "--no-use-transaction", "-uri", STANDALONE,
                               "-db", "shop", "-p", "migrations")
print("exit code:", exit_code, "| documents:", people(port=27018))
print()
print("it worked, and a failure halfway would now leave the collection half migrated")


exit code: 0 | documents: [{'greeting': 'hello ana', 'name': 'ana'}, {'greeting': 'hello bo', 'name': 'bo'}]

it worked, and a failure halfway would now leave the collection half migrated


### No error: the database called None


In [19]:
fresh_migrations()
reset_people()
write_migration("20260101000000_add_greeting.py", ADD_GREETING)

exit_code, output = run_beanie("migrate", "-uri", "mongodb://127.0.0.1:27017",
                               "-p", "migrations")                  # no -db
print("exit code:", exit_code, "| the documents in shop:", people())

with pymongo.MongoClient(URI) as client:
    names = client.list_database_names()
    print("a database called None now exists:", "None" in names)
    if "None" in names:
        print("  holding:", client["None"].list_collection_names())
        client.drop_database("None")


exit code: 0 | the documents in shop: [{'name': 'ana'}, {'name': 'bo'}]
a database called None now exists: True
  holding: ['migrations_log']


The command succeeded, the documents in `shop` are untouched, and somewhere on the server there is a
database whose name is the four characters `None`, containing a migration log that says the
migration has been applied.

Nothing about this is an error as far as the tool is concerned: a database name is a string, and
`None` formatted into a string is `"None"`. The damage is that the log is now in the wrong place, so
the real database still thinks the migration has not run, and the next attempt will run it again.


In [20]:
shutil.rmtree(WORK, ignore_errors=True)
for port in (27017, 27018):
    with pymongo.MongoClient(f"mongodb://127.0.0.1:{port}/shop") as client:
        shop = client.get_default_database()
        shop.mig_people.drop()
        shop.migrations_log.drop()
print("tidied up")


tidied up


## Recap

- A migration is a file with `Forward` and `Backward` classes. `beanie new-migration` creates it
  with both bodies empty and a timestamp in the filename, which is what orders them.
- `beanie migrate` is a subprocess. It imports your migration files in a fresh interpreter, so
  anything they import must be importable there, and models defined in a notebook cell are not.
- `@iterative_migration()` reads the function's `input_document` and `output_document` annotations,
  hands you one document at a time as two models, and writes back what changed. Both parameter names
  are required, or it raises `input_signature must not be None` at import time.
- `@free_fall_migration(document_models=[...])` gives you the collection and a session, for changes
  that are one server-side operation.
- Applied migrations are recorded in the `migrations_log` collection of the same database, which is
  what stops them running twice.
- The whole migration runs in a transaction, so it needs a replica set. `--no-use-transaction` is
  the escape and it gives up the rollback.
- Omitting `-db` migrates a database literally named `None`, with no error, leaving the real
  database's log untouched.
- `Backward` is written by you. Nothing is inferred, and it is the part everybody skips.


## What is next

**A Product Catalog** is the whole guide in one program: loaded with PyMongo, indexed for the
queries it serves, and typed with Beanie, with one line on each half saying why it got the tool it
got.


---

&#8592; **Previous:** [Link and BackLink](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/14-link-and-backlink.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
